# CrimeScope — Data sources & assumptions

**Description:** This notebook locks **scope and assumptions** before any large-scale ingestion: where and when you model (geography, time windows), how you define the **two risk targets** (recent 12‑month level vs next‑30‑day outlook), which **external sources** you will pull, and how you document **data gaps and bias**. Filling it in keeps CrimeScope’s feature pipeline, training, and demos consistent. The last cell only checks that your cluster can use the **`varanasi`** catalog.

**Workspace path:** `/Workspace/Shared/Team_varanasi/ML`  
**Catalog:** `varanasi` · **Schema:** `default` (until you add `geo`, `historical`, etc.)

**Week 1 goal:** lock these assumptions → run catalog check (below) → `02` tract geometry → `03` Chicago crime → `04` staging aggregates. Adjust any cell if your sponsor wants a different county or date range.


## 1. Geography

| Field | Week 1 default (edit if needed) |
|-------|--------------------------------|
| Primary city | Chicago, IL |
| Unit | Census tract (GEOID 11 digits) |
| Study boundary | **Cook County (FIPS 17031)** first; extend to collar counties later if needed |
| Tract vintage | **TIGER/Line 2024** (tracts for state FIPS `17`) |

**Notes:**
- Chicago lies mostly in Cook County; tract polygons come from `tl_2024_17_tract` (see notebook `02`).
- Exclude non-Illinois tracts by construction when you clip to county.

## 2. Time windows

| Field | Week 1 default |
|-------|----------------|
| Crime incidents — start / end | **2022-01-01** → *present* (Socrata `date`) |
| ACS vintage | **2022 ACS 5-year** (release 2020–2024 estimates) — update when you pull tract tables |
| Feature `as_of_date` cadence | **Monthly** snapshots for v1 |


## 3. Targets (two outputs)

| Target | Definition | Label column name (feature table) |
|--------|------------|-----------------------------------|
| **Recent level** | 12-month rolling **total incident rate** per tract (or per 1k pop) at `as_of_date` | `y_rate_12m` |
| **Next 30 days** | Count of incidents in **(as_of_date, as_of_date + 30d]** — features use data **≤ as_of_date** only | `y_next_30d_count` |

**Split person vs property later** using `primary_type` / IUCR; week 1 can use **all incidents** for pipeline smoke tests.

## 4. Source list (check when ingested)

- [x] Chicago open crime incidents — **`02`** → `varanasi.default.chicago_crimes_raw`
- [x] Staging aggregates — **`02`** → `varanasi.default.chicago_crime_monthly_by_community_area`
- [x] Census ACS 5-year (tract) — **`03`** → `varanasi.default.tract_acs_population`
- [x] TIGER / tract geometry — **`02`** → `varanasi.default.cook_tract_boundaries`
- [ ] FBI UCR or NIBRS (benchmark / sanity)
- [ ] OSM / Overpass or Overture Places (POI)
- [ ] Transit GTFS (e.g. CTA)
- [ ] Vacancy / land-use proxy
- [ ] Weather (optional)
- [ ] Live feeds (separate pipeline — do not mix into verified score without design)


## 5. Known gaps & bias

**Week 1 stance:**
- **Underreporting:** Chicago open data reflects **reported** incidents; we will not treat it as complete crime prevalence.
- **Sparse tracts:** low population tracts get unstable rates — later: MOE + **abstain** / wide intervals.
- **Community area vs tract:** notebook `04` uses **community_area** for staging until `tract_master` + point-in-polygon is ready.

_Add team notes:_
- 


In [ ]:
# Quick sanity: cluster + catalog (run after attaching a cluster)
# Use SQL to select catalog — do not set spark.databricks.sql.initial.catalog.name (not writable on many runtimes).
spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")
display(spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema"))